# Narração Wingborn — sua voz com Chatterbox

**Antes de começar (uma vez por vídeo):**
1. No Kaggle, crie um *dataset* com 2 arquivos: o `tts_plain_text.txt` (pasta `11_exports/` do projeto) e um áudio da sua voz (15–30 s, voz limpa, sem música).
2. Neste notebook: **Add Input** → escolha esse dataset.
3. **Settings → Accelerator → GPU** e **Settings → Internet → On** (a internet exige telefone verificado na conta Kaggle).

**Para gerar:** clique em **Run All**. Não é preciso reiniciar a sessão em nenhum momento.

- Teste rápido primeiro? Na célula 3, coloque `"so_primeiros": 3`.
- Não quer ficar com a aba aberta? Use **Save Version → Save & Run All (Commit)**. O Kaggle roda sozinho e os arquivos ficam guardados na aba **Output** da versão, sem prazo para baixar.

**Resultado (aba Output → Download):** `narracao_final.wav`, `narracao.srt` (legenda com o tempo de cada trecho, útil para posicionar as cenas) e `relatorio.txt` (lista os trechos que merecem ser ouvidos).


In [ ]:
# 1) Instalação (10–15 min na primeira vez). Avisos vermelhos de "dependency conflicts" são normais.
import subprocess, sys

def ok(code):
    return subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)

if ok("import chatterbox, numpy.random").returncode == 0:
    print("Chatterbox já instalado nesta sessão.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chatterbox-tts"])
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])
    # O pip costuma deixar o numpy com arquivos misturados no Kaggle: reinstala e confere em outro processo.
    for tentativa in range(1, 4):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "numpy"])
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "numpy==1.26.4"])
        r = ok("import numpy, numpy.random, chatterbox; print(numpy.__version__)")
        if r.returncode == 0:
            break
        print(f"tentativa {tentativa}: ainda com problema:", r.stderr[-300:])
    else:
        raise RuntimeError("A instalação falhou 3 vezes. Clique no botão de power (encerrar sessão), ligue de novo e rode tudo.")
    print("Instalado. Pode seguir — NÃO precisa reiniciar a sessão.")


In [ ]:
%%writefile narrar.py
# 2) Código da narração. Não precisa mexer aqui.
#!/usr/bin/env python3
"""Narra o roteiro do Wingborn Content Factory com a sua voz (Chatterbox TTS).

Pensado para o Kaggle, mas funciona em qualquer máquina com GPU:
    python -u narrar.py            (lê config.json, se existir)

Entradas (procuradas sozinhas em /kaggle/input, /content e na pasta atual):
    - tts_plain_text.txt   exportado por /exportar-projeto (ou qualquer .txt único)
    - um áudio da sua voz  (.wav .mp3 .m4a .flac .ogg .opus .aac)

Saídas na pasta atual:
    narracao_final.wav   narração completa, volume normalizado para o YouTube (-16 LUFS)
    narracao.srt         legenda com o tempo de cada trecho (serve para alinhar as cenas na edição)
    relatorio.txt        trechos, duração, take usada e trechos suspeitos
    takes/               cache: cada trecho é guardado pelo próprio conteúdo; rodar de novo só gera o que falta

Roda em processo separado do notebook de propósito: assim a instalação do Chatterbox não
exige "Restart session".
"""

import hashlib
import json
import re
import shutil
import statistics
import subprocess
import sys
import time
from pathlib import Path

DEFAULTS = {
    "exaggeration": 0.7,       # emoção: 0.5 neutro, 0.7+ dramático
    "cfg_weight": 0.3,         # menor = ritmo mais solto e menos sotaque copiado da referência
    "temperature": 0.8,
    "max_chars": 250,          # tamanho máximo de cada trecho
    "pausa_frase": 0.25,       # silêncio entre trechos do mesmo parágrafo (s)
    "pausa_paragrafo": 0.7,    # silêncio no fim de cada parágrafo (s)
    "so_primeiros": None,      # teste rápido: gera só os N primeiros trechos
    "refazer_suspeitos": 2,    # tentativas extras para trechos com duração anormal (0 desliga)
    "usar_take": {},           # {"12": 2}: força a take 2 no trecho 12
    "ref_index": 0,            # qual áudio usar se houver vários
    "ref_inicio_seg": 0,       # de onde recortar a referência
    "ref_duracao_seg": 20,     # o modelo só aproveita os primeiros ~10 s; 20 s dão margem
    "roteiro": None,           # caminho manual do .txt (opcional)
    "referencia": None,        # caminho manual do áudio (opcional)
    "zip_takes": False,        # empacotar takes/ num .zip para download
}
SEARCH_DIRS = ["/kaggle/input", "/content", "."]
AUDIO_EXTS = (".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac")
IGNORED_OUTPUTS = {"narracao_final.wav", "narracao_bruta.wav", "referencia.wav"}


class Erro(Exception):
    """Erro com mensagem pensada para o usuário do notebook."""


# ---------------------------------------------------------------------------
# Configuração e entradas
# ---------------------------------------------------------------------------

def load_config(path: Path = Path("config.json")) -> dict:
    cfg = dict(DEFAULTS)
    if path.exists():
        cfg.update(json.loads(path.read_text(encoding="utf-8")))
    cfg["usar_take"] = {int(k): int(v) for k, v in (cfg.get("usar_take") or {}).items()}
    return cfg


def _walk(dirs: list[str]) -> list[Path]:
    files = []
    for d in dirs:
        p = Path(d)
        if p.is_dir():
            files += sorted(f for f in p.rglob("*") if f.is_file() and "takes" not in f.parts)
    return files


def find_inputs(cfg: dict, dirs: list[str] = SEARCH_DIRS) -> tuple[Path, Path]:
    files = _walk(dirs)

    if cfg.get("roteiro"):
        script = Path(cfg["roteiro"])
    else:
        named = [f for f in files if f.name == "tts_plain_text.txt"]
        txts = [f for f in files if f.suffix.lower() == ".txt" and f.name not in ("relatorio.txt",)]
        if named:
            script = named[0]
        elif len(txts) == 1:
            script = txts[0]
        else:
            raise Erro(
                "Não achei o roteiro. Adicione o tts_plain_text.txt (de 11_exports/) ao seu dataset, "
                f"ou informe ROTEIRO na célula de configuração. Arquivos .txt vistos: {[str(t) for t in txts]}"
            )

    if cfg.get("referencia"):
        ref = Path(cfg["referencia"])
    else:
        audios = [f for f in files if f.suffix.lower() in AUDIO_EXTS and f.name not in IGNORED_OUTPUTS]
        if not audios:
            raise Erro("Não achei o áudio da sua voz. Adicione-o ao dataset (Add Input) ou informe REFERENCIA.")
        idx = cfg.get("ref_index", 0)
        if idx >= len(audios):
            raise Erro(f"REF_INDEX = {idx}, mas só há {len(audios)} áudio(s): {[str(a) for a in audios]}")
        if len(audios) > 1:
            print("Áudios encontrados (use REF_INDEX para escolher):")
            for i, a in enumerate(audios):
                print(f"  [{i}] {a}")
        ref = audios[idx]

    for p, nome in ((script, "roteiro"), (ref, "referência")):
        if not p.exists():
            raise Erro(f"O arquivo de {nome} não existe: {p}")
    return script, ref


# ---------------------------------------------------------------------------
# Texto
# ---------------------------------------------------------------------------

def split_long(sentence: str, max_chars: int) -> list[str]:
    """Quebra uma frase longa em pontos naturais (vírgula, ponto e vírgula, travessão), depois em espaços."""
    if len(sentence) <= max_chars:
        return [sentence]
    pieces = re.split(r"(?<=[,;:—–])\s+", sentence)
    out, cur = [], ""
    for piece in pieces:
        if len(piece) > max_chars:
            if cur:
                out.append(cur)
                cur = ""
            words, line = piece.split(), ""
            for w in words:
                if line and len(line) + 1 + len(w) > max_chars:
                    out.append(line)
                    line = w
                else:
                    line = f"{line} {w}".strip()
            cur = line
        elif cur and len(cur) + 1 + len(piece) > max_chars:
            out.append(cur)
            cur = piece
        else:
            cur = f"{cur} {piece}".strip()
    if cur:
        out.append(cur)
    return out


def split_script(text: str, max_chars: int, pausa_frase: float, pausa_paragrafo: float) -> list[tuple[str, float]]:
    """Divide em (trecho, pausa_depois). Linha em branco separa parágrafos."""
    paragraphs = [" ".join(p.split()) for p in re.split(r"\n\s*\n", text.replace("\r\n", "\n")) if p.strip()]
    chunks: list[tuple[str, float]] = []
    for p in paragraphs:
        sentences = []
        for s in re.split(r"(?<=[.!?…])\s+", p):
            sentences += split_long(s, max_chars)
        cur, start = "", len(chunks)
        for s in sentences:
            if cur and len(cur) + 1 + len(s) > max_chars:
                chunks.append((cur, pausa_frase))
                cur = s
            else:
                cur = f"{cur} {s}".strip()
        if cur:
            chunks.append((cur, pausa_frase))
        if len(chunks) > start:
            chunks[-1] = (chunks[-1][0], pausa_paragrafo)
    return chunks


# ---------------------------------------------------------------------------
# Utilidades
# ---------------------------------------------------------------------------

def file_hash(path: Path) -> str:
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def chunk_key(text: str, cfg: dict, ref_hash: str) -> str:
    """Identidade de um trecho: muda se o texto, a voz ou os ajustes de geração mudarem."""
    params = (text, ref_hash, cfg["exaggeration"], cfg["cfg_weight"], cfg["temperature"])
    return hashlib.md5(repr(params).encode("utf-8")).hexdigest()[:16]


def srt_time(seconds: float) -> str:
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def build_srt(entries: list[tuple[float, float, str]]) -> str:
    return "\n".join(
        f"{i}\n{srt_time(a)} --> {srt_time(b)}\n{text}\n" for i, (a, b, text) in enumerate(entries, 1)
    )


def trim_silence(audio, sr: int, threshold_db: float = -45.0, margin_s: float = 0.05):
    """Remove silêncio no começo e no fim do trecho, para as pausas ficarem uniformes."""
    import numpy as np

    if audio.size == 0:
        return audio
    peak = float(np.max(np.abs(audio))) or 1.0
    loud = np.where(np.abs(audio) > peak * 10 ** (threshold_db / 20))[0]
    if loud.size == 0:
        return audio
    margin = int(sr * margin_s)
    return audio[max(0, loud[0] - margin): min(len(audio), loud[-1] + 1 + margin)]


def suspicious(durations: dict[int, float], texts: dict[int, str], low: float = 0.6, high: float = 1.6) -> list[int]:
    """Trechos cuja duração foge muito da velocidade típica desta narração (corte ou alucinação)."""
    rates = [len(texts[i]) / d for i, d in durations.items() if d > 0]
    if len(rates) < 3:
        return []
    cps = statistics.median(rates)
    out = []
    for i, d in durations.items():
        expected = len(texts[i]) / cps
        if d <= 0 or d > expected * high or d < expected * low:
            out.append(i)
    return out


def prepare_reference(src: Path, inicio: float, duracao: float, out: Path = Path("referencia.wav")) -> Path:
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(inicio), "-t", str(duracao),
           "-i", str(src), "-ac", "1", str(out)]
    subprocess.run(cmd, check=True)
    return out


def normalize(src: Path, dst: Path, sr: int) -> None:
    """Volume no padrão do YouTube (~ -16 LUFS), mantendo a taxa de amostragem."""
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
                    "-af", "loudnorm=I=-16:TP=-1.5:LRA=11", "-ar", str(sr), str(dst)], check=True)


# ---------------------------------------------------------------------------
# Geração
# ---------------------------------------------------------------------------

def load_model():
    try:
        import torch
        from chatterbox.tts import ChatterboxTTS
    except Exception as e:  # noqa: BLE001 - mensagem amigável para qualquer falha de import
        raise Erro(
            "O Chatterbox não carregou. Rode de novo a célula 1 (instalação) e depois esta. "
            f"Erro original: {e!r}"
        ) from e
    if not torch.cuda.is_available():
        raise Erro("GPU desligada. No Kaggle: Settings → Accelerator → GPU T4 x2 (ou P100) e rode tudo de novo.")
    return ChatterboxTTS.from_pretrained(device="cuda")


def main() -> int:
    import numpy as np
    import soundfile as sf

    cfg = load_config()
    script_path, ref_src = find_inputs(cfg)
    print(f"Roteiro:    {script_path}")
    print(f"Referência: {ref_src}")

    text = script_path.read_text(encoding="utf-8")
    chunks = split_script(text, cfg["max_chars"], cfg["pausa_frase"], cfg["pausa_paragrafo"])
    if cfg.get("so_primeiros"):
        chunks = chunks[: int(cfg["so_primeiros"])]
        print(f"MODO TESTE: só os {len(chunks)} primeiros trechos.")
    if not chunks:
        raise Erro("O roteiro está vazio.")

    ref = prepare_reference(ref_src, cfg["ref_inicio_seg"], cfg["ref_duracao_seg"])
    ref_hash = file_hash(ref)
    texts = {i: t for i, (t, _) in enumerate(chunks, 1)}
    keys = {i: chunk_key(t, cfg, ref_hash) for i, t in texts.items()}

    takes = Path("takes")
    takes.mkdir(exist_ok=True)

    def path(i: int, t: int) -> Path:
        return takes / f"{keys[i]}_take{t}.wav"

    missing = [i for i in texts if not path(i, 1).exists()]
    print(f"{len(chunks)} trechos; {len(chunks) - len(missing)} já prontos no cache, {len(missing)} para gerar.")

    model = None
    sr = 24000

    def generate(i: int, t: int) -> None:
        nonlocal model, sr
        f = path(i, t)
        if f.exists():
            return
        if model is None:
            print("Carregando o modelo na GPU...")
            model = load_model()
            sr = model.sr
        wav = model.generate(texts[i], audio_prompt_path=str(ref), exaggeration=cfg["exaggeration"],
                             cfg_weight=cfg["cfg_weight"], temperature=cfg["temperature"])
        audio = wav.squeeze(0).detach().cpu().numpy().astype("float32")
        sf.write(str(f), trim_silence(audio, model.sr), model.sr)

    t0 = time.time()
    for n, i in enumerate(missing, 1):
        generate(i, 1)
        rest = (time.time() - t0) / n * (len(missing) - n)
        print(f"trecho {i}/{len(chunks)} pronto — faltam ~{rest / 60:.0f} min", flush=True)

    def duration(i: int, t: int) -> float:
        info = sf.info(str(path(i, t)))
        return info.frames / info.samplerate

    chosen = {i: 1 for i in texts}
    bad = suspicious({i: duration(i, 1) for i in texts}, texts)
    retries = int(cfg.get("refazer_suspeitos") or 0)
    if bad and retries:
        print(f"{len(bad)} trecho(s) com duração anormal: {bad}. Gerando até {retries} tentativa(s) extra(s)...")
        rates = [len(texts[i]) / duration(i, 1) for i in texts if i not in bad]
        cps = statistics.median(rates) if rates else 15.0
        for i in bad:
            for t in range(2, 2 + retries):
                generate(i, t)
            candidates = [t for t in range(1, 2 + retries) if path(i, t).exists()]
            expected = len(texts[i]) / cps
            chosen[i] = min(candidates, key=lambda t: abs(duration(i, t) - expected))

    for i, t in cfg["usar_take"].items():
        if i in texts:
            generate(i, t)
            chosen[i] = t

    parts, entries, cursor = [], [], 0.0
    for i, (_, pause) in enumerate(chunks, 1):
        audio, file_sr = sf.read(str(path(i, chosen[i])), dtype="float32")
        sr = file_sr
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        start = cursor
        cursor += len(audio) / sr
        entries.append((start, cursor, texts[i]))
        parts += [audio, np.zeros(int(sr * pause), dtype="float32")]
        cursor += pause
    sf.write("narracao_bruta.wav", np.concatenate(parts), sr)
    normalize(Path("narracao_bruta.wav"), Path("narracao_final.wav"), sr)
    Path("narracao.srt").write_text(build_srt(entries), encoding="utf-8")

    still_bad = suspicious({i: duration(i, chosen[i]) for i in texts}, texts)
    lines = [f"Roteiro: {script_path}", f"Referência: {ref_src}", f"Duração: {cursor / 60:.1f} min", ""]
    if still_bad:
        lines.append(f"OUÇA ESTES TRECHOS (duração ainda anormal): {still_bad}")
        lines.append("Se algum estiver ruim: coloque USAR_TAKE = {número: 2} e rode de novo (só ele é regerado).")
        lines.append("")
    for (a, b, t), i in zip(entries, texts):
        flag = "  <-- ouvir" if i in still_bad else ""
        lines.append(f"[{i:03d}] {srt_time(a)}  take {chosen[i]}  {b - a:5.1f}s  {t[:70]}{flag}")
    Path("relatorio.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
    Path("relatorio.json").write_text(json.dumps(
        {"suspeitos": still_bad, "takes": {str(i): str(path(i, chosen[i])) for i in texts},
         "textos": {str(i): texts[i] for i in texts}}, ensure_ascii=False, indent=1), encoding="utf-8")
    Path("narracao_bruta.wav").unlink()

    if cfg.get("zip_takes"):
        shutil.make_archive("takes", "zip", "takes")

    print()
    print("=" * 64)
    print(f"PRONTO: narracao_final.wav ({cursor / 60:.1f} min), narracao.srt e relatorio.txt")
    if still_bad:
        print(f"Trechos para ouvir antes de publicar: {still_bad} (veja relatorio.txt ou a célula 4)")
    else:
        print("Nenhum trecho com duração anormal.")
    print("=" * 64)
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except Erro as e:
        print(f"\nERRO: {e}")
        sys.exit(1)


In [ ]:
# 3) Ajustes (opcional). Os valores abaixo já funcionam; mude só se precisar.
import json

config = {
    "exaggeration": 0.7,        # emoção: 0.5 neutro, 0.7+ dramático (alto demais distorce)
    "cfg_weight": 0.3,          # menor = ritmo mais solto e menos sotaque copiado da referência
    "temperature": 0.8,
    "so_primeiros": None,       # TESTE RÁPIDO: coloque 3 para gerar só os 3 primeiros trechos
    "usar_take": {},            # depois de ouvir, ex.: {12: 2} usa a take 2 no trecho 12
    "refazer_suspeitos": 2,     # tentativas extras automáticas para trechos com duração estranha
    "ref_index": 0,             # se o dataset tiver vários áudios, qual usar (0 = primeiro)
    "ref_inicio_seg": 0,        # pule um início com silêncio ou ruído na sua gravação
    "roteiro": None,            # caminho manual do .txt, se a detecção automática falhar
    "referencia": None,         # caminho manual do áudio, se a detecção automática falhar
    "zip_takes": False,         # True para baixar todas as takes num .zip
}
json.dump(config, open("config.json", "w"), indent=1)
print("Ajustes salvos.")


In [ ]:
# 4) Gerar a narração. Pode rodar de novo à vontade: só o que mudou é gerado outra vez.
!python -u narrar.py


In [ ]:
# 5) (Opcional) Ouvir os trechos marcados como suspeitos no relatório.
#    Se um estiver ruim: na célula 3 coloque "usar_take": {número: 2} e rode as células 3 e 4 de novo.
import json, glob
from IPython.display import Audio, Markdown, display

MOSTRAR_TODOS = False   # True para listar todos os trechos

r = json.load(open("relatorio.json"))
alvo = list(r["textos"]) if MOSTRAR_TODOS else [str(i) for i in r["suspeitos"]]
if not alvo:
    print("Nenhum trecho suspeito. Se quiser conferir tudo, use MOSTRAR_TODOS = True.")
for i in alvo:
    display(Markdown(f"**Trecho {i}:** {r['textos'][i]}"))
    base = r["takes"][i].rsplit("_take", 1)[0]
    for f in sorted(glob.glob(base + "_take*.wav")):
        print(f.rsplit("_", 1)[1][:-4], "(em uso)" if f == r["takes"][i] else "")
        display(Audio(f))
